# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** EU General Data Protection Regulation (GDPR, Regulation (EU) 2016/679), 88 pages.
It is a good stress test: 173 recitals that paraphrase the 99 articles, dense cross-references, and a lot of
*"shall not apply where…"*, *"unless…"*, *"only where…"* clauses.

| Component | Choice |
|---|---|
| Bi-Encoder (stage 1) | `sentence-transformers/all-MiniLM-L6-v2`, cosine similarity via FAISS `IndexFlatIP` on L2-normalised vectors |
| Cross-Encoder (stage 2) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | Gemini (`gemini-2.5-flash`) if `GEMINI_API_KEY` is set, otherwise local `Qwen/Qwen2.5-3B-Instruct` (greedy decoding) |

Both pipelines use **the same chunks, the same Top-10 candidates and the same prompt**. The only difference is
which 3 chunks reach the LLM.

## 1. Setup

In [1]:
import os, re, time, textwrap, urllib.request
import numpy as np, pandas as pd, torch, faiss, pypdf
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)
torch.manual_seed(0)

PDF_URL  = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679"
PDF_PATH = "data/gdpr.pdf"

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOCAL_LLM     = "Qwen/Qwen2.5-3B-Instruct"
GEMINI_MODEL  = "gemini-2.5-flash"

CHUNK_WORDS, OVERLAP_WORDS = 120, 30   # ~160 tokens per chunk, fits the 256/512-token limits of both encoders
TOP_K, TOP_N = 10, 3                   # retrieve 10 candidates, send 3 to the LLM

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

W1019 09:29:58.009000 55850 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps


## 2. Load and clean the PDF

In [2]:
if not os.path.exists(PDF_PATH):
    os.makedirs("data", exist_ok=True)
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(PDF_PATH, "wb") as f:
        f.write(r.read())

reader = pypdf.PdfReader(PDF_PATH)
pages = []
for i, page in enumerate(reader.pages):
    t = page.extract_text()
    t = re.sub(r"-\s*\n\s*", "", t)                                              # re-join hyphenated line breaks
    t = re.sub(r"4\.5\.2016 L 119/\d+ Official Journal of the European Union EN", "", t)  # running page header
    t = re.sub(r"\s+", " ", t).strip()
    pages.append((i + 1, t))

print(len(pages), "pages,", sum(len(t.split()) for _, t in pages), "words")
print(pages[40][1][:400])

88 pages, 54927 words
(d) where the processing is based on point (f) of Article 6(1), the legitimate interests pursued by the controller or by a third party; (e) the recipients or categories of recipients of the personal data, if any; (f) where applicable, the fact that the controller intends to transfer personal data to a third country or international organisation and the existence or absence of an adequacy decision 


## 3. Chunking

Fixed-size sliding windows of 120 words with 30 words of overlap, keeping the page number of the first word.
This is deliberately *structure-agnostic* (a common default), so a single article is often split across chunks,
and an exception paragraph (e.g. Art. 17(3)) usually lands in a different chunk from the rule it qualifies.

In [3]:
words = [(w, pg) for pg, t in pages for w in t.split(" ")]
chunks = []
for start in range(0, len(words), CHUNK_WORDS - OVERLAP_WORDS):
    win = words[start:start + CHUNK_WORDS]
    chunks.append({"id": len(chunks), "page": win[0][1], "text": " ".join(w for w, _ in win)})

print(len(chunks), "chunks")
print(chunks[305])

611 chunks
{'id': 305, 'page': 37, 'text': "on consent to the processing of personal data that is not necessary for the performance of that contract. Article 8 Conditions applicable to child's consent in relation to information society services 1. Where point (a) of Article 6(1) applies, in relation to the offer of information society services directly to a child, the processing of the personal data of a child shall be lawful where the child is at least 16 years old. Where the child is below the age of 16 years, such processing shall be lawful only if and to the extent that consent is given or authorised by the holder of parental responsibility over the child. Member States may provide by law for a lower age for"}


## 4. Stage 1 — Bi-Encoder index

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=64,
                              normalize_embeddings=True, show_progress_bar=False)
index_time = time.perf_counter() - t0

# Inner product on unit vectors == cosine similarity
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb.astype(np.float32))
print(f"indexed {index.ntotal} chunks (dim={chunk_emb.shape[1]}) in {index_time:.2f}s (one-off, offline cost)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

indexed 611 chunks (dim=384) in 1.64s (one-off, offline cost)


## 5. Stage 2 — Cross-Encoder

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 6. Retrieval pipelines

In [6]:
def baseline_retrieve(query, k=TOP_K):
    # Pipeline 1: embed the query once, cosine top-k. Returns [(chunk_id, cosine)], best first.
    q = bi_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

def rerank(query, candidates):
    # Pipeline 2: score every (query, chunk) pair jointly, re-sort by cross-encoder score.
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    order = np.argsort(-ce_scores)
    return [(candidates[i][0], float(ce_scores[i])) for i in order]

def full_bi_rank(query):
    # Rank of every chunk in the whole corpus under the bi-encoder (used for analysis only).
    q = bi_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False)
    sims = chunk_emb @ q[0]
    return {int(cid): r + 1 for r, cid in enumerate(np.argsort(-sims))}

## 7. LLM and the shared prompt

The prompt is identical for both pipelines; only `{context}` changes. Decoding is greedy so differences between
Answer 1 and Answer 2 come from the retrieved context, not sampling noise.

In [7]:
PROMPT = '''You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context excerpts below. Quote exact figures, conditions and exceptions where
relevant. If the context does not contain the answer, say "The provided context does not contain the answer."

Context:
{context}

Question: {question}

Answer (2-4 sentences):'''

def build_context(ranked):
    return "\n\n".join(f"[Excerpt {i+1} | page {chunks[cid]['page']}]\n{chunks[cid]['text']}"
                       for i, (cid, _) in enumerate(ranked[:TOP_N]))

if os.environ.get("GEMINI_API_KEY"):
    from google import genai
    _client = genai.Client()
    LLM_NAME = GEMINI_MODEL
    def generate(prompt):
        r = _client.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                                            config={"temperature": 0.0})
        return r.text.strip()
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _tok = AutoTokenizer.from_pretrained(LOCAL_LLM)
    _llm = AutoModelForCausalLM.from_pretrained(LOCAL_LLM, dtype=torch.float16).to(DEVICE).eval()
    LLM_NAME = LOCAL_LLM
    @torch.inference_mode()
    def generate(prompt):
        x = _tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(DEVICE)
        out = _llm.generate(**x, max_new_tokens=220, do_sample=False)
        return _tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("LLM:", LLM_NAME)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM: Qwen/Qwen2.5-3B-Instruct


## 8. Failure-mode queries

Each query is paired with a short **evidence phrase** copied from the article that actually answers it. The chunk
containing that phrase is the *critical chunk*; we track its rank before and after re-ranking. (The phrase is used
only for analysis — never by the retrievers.)

In [8]:
FAILURE_QUERIES = [
    dict(category="Keyword trap / distractor",
         query="What is the maximum fine for infringing the basic principles for processing, including conditions for consent?",
         evidence="up to 20 000 000 EUR, or in the case of an undertaking, up to 4 %",
         source="Art. 83(5)"),
    dict(category="Exceptions & corner cases",
         query="When does the right to erasure not apply?",
         evidence="Paragraphs 1 and 2 shall not apply to the extent that processing is necessary",
         source="Art. 17(3)"),
    dict(category="Negation & constraints",
         query="Is profiling of individuals permitted?",
         evidence="right not to be subject to a decision based solely on automated processing",
         source="Art. 22(1)-(2)"),
    dict(category="Exceptions & corner cases (role limit)",
         query="Who is liable when a processor causes damage?",
         evidence="A processor shall be liable for the damage caused by processing only where",
         source="Art. 82(2)"),
    dict(category="Buried at rank 4-10",
         query="When can a data subject restrict processing?",
         evidence="the accuracy of the personal data is contested by the data subject",
         source="Art. 18(1)"),
    dict(category="Buried at rank 4-10",
         query="Can a controller transfer data abroad for important reasons of public interest?",
         evidence="important reasons of public interest",
         source="Recital 112 / Art. 49(1)(d)"),
]

# Control queries: easy questions where the baseline is already right, plus one recall-ceiling case
CONTROL_QUERIES = [
    dict(category="Control (easy)", query="At what age can a child consent to online services?",
         evidence="at least 16 years old", source="Art. 8(1)"),
    dict(category="Recall ceiling", query="Is a decision based solely on automated processing allowed?",
         evidence="shall not be based on special categories", source="Art. 22(4)"),
]

def critical_chunks(evidence):
    return [c["id"] for c in chunks if evidence.lower() in c["text"].lower()]

## 9. Run both pipelines and record rank shifts

In [9]:
def run(q):
    base = baseline_retrieve(q["query"])
    rr = rerank(q["query"], base)
    bi_full = full_bi_rank(q["query"])
    gold = critical_chunks(q["evidence"])
    bi_pos = {cid: i + 1 for i, (cid, _) in enumerate(base)}
    ce_pos = {cid: i + 1 for i, (cid, _) in enumerate(rr)}
    # The best-placed critical chunk under each pipeline
    g = min(gold, key=lambda c: (ce_pos.get(c, 99), bi_full[c]))
    return dict(**q, base=base, rr=rr, gold=g,
                bi_rank=bi_full[g], ce_rank=ce_pos.get(g),
                in_top3_base=any(bi_pos.get(c, 99) <= TOP_N for c in gold),
                in_top3_rr=any(ce_pos.get(c, 99) <= TOP_N for c in gold))

results = [run(q) for q in FAILURE_QUERIES + CONTROL_QUERIES]

rank_table = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "source": r["source"],
    "critical chunk": f"#{r['gold']} (p.{chunks[r['gold']]['page']})",
    "bi-encoder rank": r["bi_rank"],
    "cross-encoder rank": r["ce_rank"] if r["ce_rank"] else "not in top-10",
    "shift": f"#{r['bi_rank']} → #{r['ce_rank']}" if r["ce_rank"] else f"#{r['bi_rank']} (never retrieved)",
    "in top-3 (P1)": r["in_top3_base"], "in top-3 (P2)": r["in_top3_rr"],
} for r in results])
rank_table

,category,query,source,critical chunk,bi-encoder rank,cross-encoder rank,shift,in top-3 (P1),in top-3 (P2)
0,Keyword trap / distractor,"What is the maximum fine for infringing the basic principles for processing, including conditions for consent?",Art. 83(5),#582 (p.82),4,1,#4 → #1,False,True
1,Exceptions & corner cases,When does the right to erasure not apply?,Art. 17(3),#346 (p.44),4,2,#4 → #2,False,True
2,Negation & constraints,Is profiling of individuals permitted?,Art. 22(1)-(2),#357 (p.46),5,2,#5 → #2,False,True
3,Exceptions & corner cases (role limit),Who is liable when a processor causes damage?,Art. 82(2),#574 (p.81),5,1,#5 → #1,False,True
4,Buried at rank 4-10,When can a data subject restrict processing?,Art. 18(1),#348 (p.44),5,2,#5 → #2,False,True
5,Buried at rank 4-10,Can a controller transfer data abroad for important reasons of public interest?,Recital 112 / Art. 49(1)(d),#178 (p.21),8,2,#8 → #2,False,True
6,Control (easy),At what age can a child consent to online services?,Art. 8(1),#305 (p.37),1,1,#1 → #1,True,True
7,Recall ceiling,Is a decision based solely on automated processing allowed?,Art. 22(4),#358 (p.46),217,not in top-10,#217 (never retrieved),False,False


### Full Top-10 view (bi-encoder order vs cross-encoder order)

In [10]:
def show_ranks(r):
    ce_pos = {cid: i + 1 for i, (cid, _) in enumerate(r["rr"])}
    ce_sc = dict(r["rr"])
    gold = set(critical_chunks(r["evidence"]))
    rows = [{"bi rank": i + 1, "ce rank": ce_pos[cid], "chunk": cid, "page": chunks[cid]["page"],
             "cosine": round(s, 3), "ce score": round(ce_sc[cid], 2), "critical": "★" if cid in gold else "",
             "text": chunks[cid]["text"][:110] + "…"} for i, (cid, s) in enumerate(r["base"])]
    print(f"\n=== [{r['category']}] {r['query']}")
    display(pd.DataFrame(rows).set_index("bi rank"))

for r in results[:len(FAILURE_QUERIES)]:
    show_ranks(r)


=== [Keyword trap / distractor] What is the maximum fine for infringing the basic principles for processing, including conditions for consent?


,ce rank,chunk,page,cosine,ce score,critical,text
bi rank,,,,,,,
1,6,587,83,0.609,-3.36,,Member States shall lay down the rules on other penalties applicable to infringements of this Regulation in pa…
2,4,234,27,0.604,-2.83,,of a minor infringement or if the fine likely to be imposed would constitute a disproportionate burden to a na…
3,2,581,82,0.598,0.80,,"or losses avoided, directly or indirectly, from the infringement. 3. If a controller or processor intentionall…"
4,1,582,82,0.585,4.00,★,"preceding financial year, whichever is higher: (a) the obligations of the controller and the processor pursuan…"
5,5,578,82,0.579,-3.08,,to in points (a) to (h) and (j) of Article 58(2). When deciding whether to impose an administrative fine and d…
6,7,237,27,0.572,-5.16,,"jurisdiction and the recognition and enforcement of judgments in civil and commercial matters (OJ L 351, 20.12…"
7,8,62,8,0.571,-5.59,,of that specific situation. Consent is presumed not to be freely given if it does not allow separate consent t…
8,10,586,83,0.569,-8.36,,a manner that the fine is initiated by the competent supervisory authority and imposed by competent national c…
9,9,235,27,0.568,-5.89,,code of conduct and any other aggravating or mitigating factor. The imposition of penalties including administ…



=== [Exceptions & corner cases] When does the right to erasure not apply?


,ce rank,chunk,page,cosine,ce score,critical,text
bi rank,,,,,,,
1,1,343,43,0.640,3.52,,"have incomplete personal data completed, including by means of providing a supplementary statement. Article 17…"
2,4,104,13,0.624,0.92,,"expression and information, for compliance with a legal obligation, for the performance of a task carried out …"
3,3,342,43,0.585,1.51,,"requested by the data subject, the information shall be provided in a commonly used electronic form. 4. The ri…"
4,2,346,44,0.578,2.76,★,"the erasure by such controllers of any links to, or copy or replication of, those personal data. 3. Paragraphs…"
5,7,345,44,0.487,-2.76,,obligation in Union or Member State law to which the controller is subject; (f) the personal data have been co…
6,10,101,12,0.478,-8.95,,information or processing activities to which the request relates. (64) The controller should use all reasonab…
7,5,110,13,0.470,0.34,,data subject to obtain the erasure of personal data and the limitations of that right as set out in this Regul…
8,8,56,7,0.469,-4.23,,"for which they are processed. This requires, in particular, ensuring that the period for which the personal da…"
9,6,109,13,0.459,-0.07,,of an official authority vested in the controller. The data subject's right to transmit or receive personal da…



=== [Negation & constraints] Is profiling of individuals permitted?


...

,ce rank,chunk,page,cosine,ce score,critical,text
bi rank,,,,,,,
1,2,575,81,0.748,6.05,,processors or where it has acted outside or contrary to lawful instructions of the controller. 3. A controller…
2,3,230,27,0.661,5.52,,unless the controller is a public authority of a Member State acting in the exercise of its public powers. (14…
3,4,576,81,0.661,4.75,,held liable for the entire damage in order to ensure effective compensation of the data subject. 5. Where a co…
4,6,231,27,0.658,3.02,,the objectives of this Regulation. This is without prejudice to any claims for damage deriving from the violat…
5,1,574,81,0.610,6.39,★,permits the consolidation thereof. Article 82 Right to compensation and liability 1. Any person who has suffer…
6,8,578,82,0.541,-2.04,,to in points (a) to (h) and (j) of Article 58(2). When deciding whether to impose an administrative fine and d…
7,5,232,27,0.540,4.05,,"entire damage. However, where they are joined to the same judicial proceedings, in accordance with Member Stat…"
8,7,579,82,0.534,0.37,,taken by the controller or processor to mitigate the damage suffered by data subjects; (d) the degree of respo…
9,9,580,82,0.498,-3.78,,"infringement became known to the supervisory authority, in particular whether, and if so to what extent, the c…"



=== [Buried at rank 4-10] When can a data subject restrict processing?


,ce rank,chunk,page,cosine,ce score,critical,text
bi rank,,,,,,,
1,1,349,44,0.742,5.64,,"but they are required by the data subject for the establishment, exercise or defence of legal claims; (d) the …"
2,4,308,38,0.706,2.73,,(a) the data subject has given explicit consent to the processing of those personal data for one or more speci…
3,5,112,13,0.670,1.76,,or the fundamental rights and freedoms of the data subject. (70) Where personal data are processed for the pur…
4,7,299,36,0.669,-0.13,,"is subject. The purpose of the processing shall be determined in that legal basis or, as regards the processin…"
5,2,348,44,0.663,5.09,★,of processing 1. The data subject shall have the right to obtain from the controller restriction of processing…
6,3,106,13,0.659,4.80,,"include, inter alia, temporarily moving the selected data to another processing system, making the selected pe…"
7,6,357,46,0.657,0.55,,"or her, unless the processing is necessary for the performance of a task carried out for reasons of public int…"
8,8,302,37,0.650,-0.32,,"processing; (b) the context in which the personal data have been collected, in particular regarding the relati…"
9,9,501,69,0.650,-0.93,,issue warnings to a controller or processor that intended processing operations are likely to infringe provisi…



=== [Buried at rank 4-10] Can a controller transfer data abroad for important reasons of public interest?


,ce rank,chunk,page,cosine,ce score,critical,text
bi rank,,,,,,,
1,3,181,21,0.698,4.53,,"limited number of data subjects, could also be possible for the purposes of the compelling legitimate interest…"
2,5,473,64,0.661,3.58,,"may take place only if the transfer is not repetitive, concerns only a limited number of data subjects, is nec…"
3,9,161,19,0.638,0.93,,"controllers, processors in the same or another third country or international organisation. In any event, tran…"
4,7,182,21,0.614,1.30,,"the country of final destination, and should provide suitable safeguards to protect fundamental rights and fre…"
5,8,325,40,0.604,1.13,,"applicable, of the controller's representative; (b) the contact details of the data protection officer, where …"
6,1,180,21,0.591,6.41,,such provisions to the Commission. Any transfer to an international humanitarian organisation of personal data…
7,4,470,64,0.591,4.45,★,"consented to the proposed transfer, after having been informed of the possible risks of such transfers for the…"
8,2,178,21,0.584,4.89,★,"those persons or, if they are to be the recipients, taking into full account the interests and fundamental rig…"
9,10,341,43,0.582,0.91,,of such processing for the data subject. 2. Where personal data are transferred to a third country or to an in…


## 10. Generate Answer 1 (baseline) vs Answer 2 (re-ranked)

In [11]:
for r in results:
    r["answer1"] = generate(PROMPT.format(context=build_context(r["base"]), question=r["query"]))
    r["answer2"] = generate(PROMPT.format(context=build_context(r["rr"]), question=r["query"]))
    print("=" * 110)
    print(f"[{r['category']}] {r['query']}   (ground truth: {r['source']})")
    print(f"P1 top-3 chunks: {[c for c, _ in r['base'][:3]]}   P2 top-3 chunks: {[c for c, _ in r['rr'][:3]]}")
    print("\nANSWER 1 (baseline):\n" + textwrap.fill(r["answer1"], 110))
    print("\nANSWER 2 (re-ranked):\n" + textwrap.fill(r["answer2"], 110))

[Keyword trap / distractor] What is the maximum fine for infringing the basic principles for processing, including conditions for consent?   (ground truth: Art. 83(5))
P1 top-3 chunks: [587, 234, 581]   P2 top-3 chunks: [582, 581, 80]

ANSWER 1 (baseline):
The maximum fine for infringing the basic principles for processing, including conditions for consent, is up
to 10,000,000 EUR or 2% of the total worldwide annual turnover of the preceding financial year, whichever is
higher.

ANSWER 2 (re-ranked):
The maximum fine for infringing the basic principles for processing, including conditions for consent, is up
to 20,000,000 EUR, or in the case of an undertaking, up to 4% of the total worldwide annual turnover of the
preceding financial year, whichever is higher.


[Exceptions & corner cases] When does the right to erasure not apply?   (ground truth: Art. 17(3))
P1 top-3 chunks: [343, 104, 342]   P2 top-3 chunks: [343, 346, 342]

ANSWER 1 (baseline):
The right to erasure (also known as the 'right to be forgotten') does not apply when the personal data are no
longer necessary in relation to the purposes for which they were collected or otherwise processed, or when the
data subject has withdrawn consent on which the processing is based.

ANSWER 2 (re-ranked):
The right to erasure (right to be forgotten) does not apply when processing is necessary for exercising the
right of freedom of expression and information, for compliance with a legal obligation, for reasons of public
interest in the area of public health, or for archiving purposes.


[Negation & constraints] Is profiling of individuals permitted?   (ground truth: Art. 22(1)-(2))
P1 top-3 chunks: [117, 354, 113]   P2 top-3 chunks: [117, 357, 113]

ANSWER 1 (baseline):
Profiling is allowed under specific conditions as outlined in the regulation, but it must comply with the
rules governing the processing of personal data, such as the legal grounds for processing and data protection
principles. The European Data Protection Board can provide guidance on this context.

ANSWER 2 (re-ranked):
Profiling is subject to the rules of this Regulation governing the processing of personal data, such as the
legal grounds for processing or data protection principles. It should be allowed only under specific
conditions, as determined by the European Data Protection Board (EDPB).


[Exceptions & corner cases (role limit)] Who is liable when a processor causes damage?   (ground truth: Art. 82(2))
P1 top-3 chunks: [575, 230, 576]   P2 top-3 chunks: [574, 575, 230]

ANSWER 1 (baseline):
A processor is liable for the entire damage caused if it is responsible for the damage and cannot prove it was
not in any way responsible for the event giving rise to the damage. If there are multiple controllers or
processors involved, each is held liable for the entire damage to ensure effective compensation of the data
subject.

ANSWER 2 (re-ranked):
A processor shall be liable for the damage caused by processing only where it has not complied with
obligations of this Regulation specifically directed to processors or where it has acted outside or contrary
to lawful instructions of the controller. If the processor can prove that it is not in any way responsible for
the event giving rise to the damage, it may also be exempt from liability.


[Buried at rank 4-10] When can a data subject restrict processing?   (ground truth: Art. 18(1))
P1 top-3 chunks: [349, 308, 112]   P2 top-3 chunks: [349, 348, 106]

ANSWER 1 (baseline):
A data subject can restrict processing if they have given explicit consent to the processing of their personal
data for one or more specified purposes, or if processing is necessary for the purposes of carrying out the
obligations and exercising specific rights of the controller or of the data subject in the field of employment
and social security and social protection law, as authorized by Union or Member State law or a collective
agreement. Additionally, processing can be restricted if the data subject has objected to processing pursuant
to Article 21(1) pending the verification whether the legitimate grounds of the controller override those of
the data subject.

ANSWER 2 (re-ranked):
A data subject can restrict processing when one of the following applies: the accuracy of the personal data is
contest

[Buried at rank 4-10] Can a controller transfer data abroad for important reasons of public interest?   (ground truth: Recital 112 / Art. 49(1)(d))
P1 top-3 chunks: [181, 473, 161]   P2 top-3 chunks: [180, 178, 181]

ANSWER 1 (baseline):
The provided context does not explicitly mention data transfers for important reasons of public interest.
However, it states that data transfers can occur under certain conditions, such as when they are necessary for
compelling legitimate interests pursued by the controller, which are not overridden by the interests or rights
and freedoms of the data subject. These conditions require the controller to assess all circumstances
surrounding the data transfer and provide suitable safeguards.

ANSWER 2 (re-ranked):
Yes, a controller can transfer data abroad for important reasons of public interest, such as in cases of
international data exchange between competition authorities, tax or customs administrations, financial
supervisory authorities, services comp

[Control (easy)] At what age can a child consent to online services?   (ground truth: Art. 8(1))
P1 top-3 chunks: [305, 306, 91]   P2 top-3 chunks: [305, 306, 53]

ANSWER 1 (baseline):
A child can consent to the processing of their personal data for information society services offered directly
to them only if they are at least 16 years old. If the child is below 16 years old, consent must be given or
authorised by the holder of parental responsibility over the child, with member states allowed to set a lower
age not below 13 years.

ANSWER 2 (re-ranked):
A child can consent to the processing of their personal data for information society services directly offered
to them if they are at least 16 years old. For those under 16, consent must be given or authorised by the
holder of parental responsibility over the child, unless a lower age is set by law, which cannot be below 13
years.


[Recall ceiling] Is a decision based solely on automated processing allowed?   (ground truth: Art. 22(4))
P1 top-3 chunks: [357, 356, 106]   P2 top-3 chunks: [357, 113, 328]

ANSWER 1 (baseline):
No, decisions based solely on automated processing are not allowed unless they are necessary for entering into
or performing a contract with the data subject, or are authorised by Union or Member State law that also
provides suitable measures to safeguard the data subject's rights.

ANSWER 2 (re-ranked):
A decision based solely on automated processing is not allowed unless it is necessary for entering into or
performing a contract with the data subject, or is authorised by Union or Member State law that also provides
suitable measures to safeguard the data subject's rights.


In [12]:
# Ground-truth text of the critical chunk for each failure query, for checking the answers above
for r in results[:len(FAILURE_QUERIES)]:
    print(f"--- {r['source']} (chunk {r['gold']}):\n{textwrap.fill(chunks[r['gold']]['text'], 110)}\n")

--- Art. 83(5) (chunk 582):
preceding financial year, whichever is higher: (a) the obligations of the controller and the processor
pursuant to Articles 8, 11, 25 to 39 and 42 and 43; (b) the obligations of the certification body pursuant to
Articles 42 and 43; (c) the obligations of the monitoring body pursuant to Article 41(4). 5. Infringements of
the following provisions shall, in accordance with paragraph 2, be subject to administrative fines up to 20
000 000 EUR, or in the case of an undertaking, up to 4 % of the total worldwide annual turnover of the
preceding financial year, whichever is higher: (a) the basic principles for processing, including conditions
for consent, pursuant to Articles 5, 6, 7 and 9; (b) the data

--- Art. 17(3) (chunk 346):
the erasure by such controllers of any links to, or copy or replication of, those personal data. 3. Paragraphs
1 and 2 shall not apply to the extent that processing is necessary: (a) for exercising the right of freedom of
expression and i

## 11. Latency profiling

Each stage is warmed up, then timed over every query × `REPEATS` runs. Retrieval times exclude the one-off
indexing cost. LLM time is measured separately because it is the same for both pipelines (same prompt size).

In [13]:
REPEATS = 20
all_queries = [q["query"] for q in FAILURE_QUERIES + CONTROL_QUERIES]
for q in all_queries[:2]:                       # warm-up (MPS kernels, tokenizer caches)
    rerank(q, baseline_retrieve(q))

def timeit(fn):
    ts = []
    for _ in range(REPEATS):
        for q in all_queries:
            t0 = time.perf_counter(); fn(q); ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)

cands = {q: baseline_retrieve(q) for q in all_queries}
t_base   = timeit(baseline_retrieve)                         # encode query + FAISS search
t_rerank = timeit(lambda q: rerank(q, cands[q]))             # 10 cross-encoder pairs only
t_p2     = timeit(lambda q: rerank(q, baseline_retrieve(q))) # full pipeline 2 retrieval

t_llm = []
for q in all_queries[:4]:
    p = PROMPT.format(context=build_context(cands[q]), question=q)
    t0 = time.perf_counter(); generate(p); t_llm.append((time.perf_counter() - t0) * 1000)
t_llm = np.array(t_llm)

def row(name, a): return {"stage": name, "mean ms": a.mean(), "median ms": np.median(a), "p95 ms": np.percentile(a, 95)}
latency = pd.DataFrame([
    row("P1 retrieval: bi-encoder query embed + FAISS top-10", t_base),
    row("Cross-encoder re-rank of 10 pairs", t_rerank),
    row("P2 retrieval: bi-encoder + cross-encoder", t_p2),
    row(f"LLM generation ({LLM_NAME}), either pipeline", t_llm),
]).round(2)
latency["P2 / P1"] = ""
latency.loc[2, "P2 / P1"] = f"{t_p2.mean() / t_base.mean():.1f}x"
latency

,stage,mean ms,median ms,p95 ms,P2 / P1
0,P1 retrieval: bi-encoder query embed + FAISS top-10,8.30,7.43,10.24,
1,Cross-encoder re-rank of 10 pairs,53.42,52.82,81.03,
2,P2 retrieval: bi-encoder + cross-encoder,58.88,57.60,68.17,7.1x
3,"LLM generation (Qwen/Qwen2.5-3B-Instruct), either pipeline",9678.41,9569.57,10855.15,


In [14]:
e2e = pd.DataFrame({
    "pipeline": ["P1 baseline (retrieval + LLM)", "P2 re-rank (retrieval + rerank + LLM)"],
    "mean end-to-end ms": [t_base.mean() + t_llm.mean(), t_p2.mean() + t_llm.mean()],
})
e2e["re-rank overhead %"] = ["", f"{100 * (t_p2.mean() - t_base.mean()) / (t_base.mean() + t_llm.mean()):.1f}%"]
e2e.round(1)

,pipeline,mean end-to-end ms,re-rank overhead %
0,P1 baseline (retrieval + LLM),9686.7,
1,P2 re-rank (retrieval + rerank + LLM),9737.3,0.5%


## 12. Scaling: why not run the Cross-Encoder over the whole corpus?

Measure what it actually costs to cross-encode **every** chunk for one query, and extrapolate linearly to bigger
collections. The bi-encoder side stays roughly constant per query because chunk embeddings are pre-computed once.

In [15]:
q = FAILURE_QUERIES[0]["query"]
all_pairs = [(q, c["text"]) for c in chunks]
cross_encoder.predict(all_pairs[:64], batch_size=64, show_progress_bar=False)  # warm-up
t0 = time.perf_counter()
full_scores = cross_encoder.predict(all_pairs, batch_size=64, show_progress_bar=False)
t_full = (time.perf_counter() - t0) * 1000
per_pair = t_full / len(chunks)

full_order = np.argsort(-full_scores)
gold = results[0]["gold"]
print(f"Cross-encoding all {len(chunks)} chunks: {t_full:.0f} ms ({per_pair:.2f} ms/pair) "
      f"vs {t_rerank.mean():.0f} ms for 10 pairs")
print(f"Critical chunk rank under full cross-encoder scan: #{int(np.where(full_order == gold)[0][0]) + 1}")

scale = pd.DataFrame({"corpus size (chunks)": [len(chunks), 10_000, 1_000_000, 100_000_000]})
scale["cross-encoder full scan / query"] = [f"{n * per_pair / 1000:,.1f} s" for n in scale["corpus size (chunks)"]]
scale["bi-encoder + rerank top-10 / query"] = f"~{t_p2.mean():.0f} ms (ANN index; near-constant)"
scale

Cross-encoding all 611 chunks: 4359 ms (7.13 ms/pair) vs 53 ms for 10 pairs
Critical chunk rank under full cross-encoder scan: #1


,corpus size (chunks),cross-encoder full scan / query,bi-encoder + rerank top-10 / query
0,611,4.4 s,~59 ms (ANN index; near-constant)
1,10000,71.3 s,~59 ms (ANN index; near-constant)
2,1000000,"7,134.4 s",~59 ms (ANN index; near-constant)
3,100000000,"713,443.1 s",~59 ms (ANN index; near-constant)


## 13. Summary

* §9 shows the main result. On all six failure queries the critical chunk was at bi-encoder rank 4–8, outside
  the LLM's context, and the cross-encoder moved it into the top 3 (to #1 or #2).
* §10: in 5 of the 6 queries this changed the answer in a meaningful way (wrong fine amount fixed, wrong exceptions
  replaced, a "role limit" restored, a non-answer turned into an answer). For *profiling*, Art. 22 reached the
  context but the 3B LLM still paraphrased the recital, so better retrieval is necessary but not sufficient.
* Control queries: when the baseline is already right, re-ranking keeps it right. When the critical chunk never
  enters the bi-encoder top 10 (Art. 22(4), rank #217), re-ranking cannot recover it. Stage-1 recall limits
  stage-2 precision.
* Re-ranking adds about 50 ms per query, about 0.5% of end-to-end latency with a local LLM.
* See `REPORT.md` for the written analysis.